<a href="https://colab.research.google.com/github/NahomKidane/ai301-labs/blob/main/m04-classification/guided-labs/AI301_Guided_Lab_Part2_Library_Classification_With_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI 301 Guided Lab: Classification, Part 2 · Classification With Embeddings

**Module 4 · Classification**

> **File > Save a copy in Drive** before you start. Switch the runtime on: **Runtime > Change
> runtime type > T4 GPU**. This notebook embeds about ten thousand sentences, which is slow on a
> CPU. It downloads one embedding model of roughly 420 MB and one dataset of roughly 1 MB.

Part 1 used a model that already knew how to classify. This notebook uses a model that knows
nothing about sentiment. It turns each review into a vector, and a separate classifier learns the
sentiment from those vectors.

Four ways to get from vectors to a label appear here, in order of how much they need from you:

- A logistic regression trained on the vectors.
- No classifier at all, just the average vector of each class.
- Two sentences describing the classes, and no labels at all.
- A nearest neighbours vote.

In [ ]:
!pip install -q transformers datasets sentence-transformers

---

## Part 0 · Setup

This is a separate notebook, so the data and the evaluation helper from Part 1 have to be loaded
again. Nothing here is new.

In [ ]:
# quiet the loading reports and the progress bar warning, neither is an error
import warnings
warnings.filterwarnings("ignore")

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# the Hub needs the full namespace and name, the bare "rotten_tomatoes" no longer resolves
data = load_dataset("cornell-movie-review-data/rotten_tomatoes")


def evaluate(y_true, y_pred, class_names=("negative", "positive"), title=""):
    """Print the per class scores and draw the confusion matrix as counts and as percentages."""
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

    counts = confusion_matrix(y_true, y_pred)
    rates = confusion_matrix(y_true, y_pred, normalize="true")

    figure, axes = plt.subplots(1, 2, figsize=(11, 4))

    ConfusionMatrixDisplay(counts, display_labels=class_names).plot(
        ax=axes[0], cmap="Blues", colorbar=False
    )
    axes[0].set_title("Counts")

    ConfusionMatrixDisplay(rates, display_labels=class_names).plot(
        ax=axes[1], cmap="Blues", colorbar=False, values_format=".0%"
    )
    axes[1].set_title("Percent of each true class")

    if title:
        figure.suptitle(title)

    plt.tight_layout()
    plt.show()


print(data)

---

## Part 1 · Two steps instead of one

Part 1 was one step. Text went into a model and a label came out, and the model had been trained
for that exact job.

This route splits the work in two. A frozen embedding model turns text into vectors, and a separate
classifier is trained on those vectors. The embedding model is never updated. It has no idea what
the task is.

That split is why the route is useful. The expensive half, the model that understands language, is
downloaded and reused. The cheap half, a classifier over a few hundred numbers, is trained on your
own labels in seconds on a CPU.

There are two ways to get one vector for a whole review, and this notebook takes the shorter one.

The long way is to run a plain BERT yourself, tokenize, pad, build the attention mask, take the
output, and pull out one vector to stand for the sentence, usually the `[CLS]` position. Jay
Alammar's [A Visual Guide to Using BERT for the First
Time](https://jalammar.github.io/a-visual-guide-to-using-bert-for-the-first-time/) does exactly
that, step by step with a picture for each one. Read it to see what the single call below is doing
underneath.

The short way is a model where that step is already part of the model and was trained for it. That
is what a sentence transformer is. Module 3 laid out the choices, the `[CLS]` vector, the mean of
the token vectors, or the element wise max, and said none of them is obviously right. The model
below has made that choice, and made it during training rather than after.

The difference is not cosmetic. A `[CLS]` vector taken from a model that was never trained to
produce sentence vectors is a weaker representation than one from a model that was.

- **Python note:** `SentenceTransformer` wraps a transformer so that `.encode` returns one vector
  per input string rather than one vector per token. The pooling that turns token vectors into a
  sentence vector happens inside it.

> **Predict before running.**
>
> The embedding model was trained on general text and has never seen this dataset.
>
> 1. Will the vectors for two positive reviews be closer to each other than to a negative review,
>    given that nothing told the model what positive means?
> 2. What would have to be true of the training text for that to happen?

**Your prediction:**

_Type here._

> Model card: [sentence-transformers/all-mpnet-base-v2](https://huggingface.co/sentence-transformers/all-mpnet-base-v2).
> Read it for what the model was trained on and what its output vector represents, since both
> decide whether the classifier below has anything to work with.
>
> Library reference: [SentenceTransformer](https://sbert.net/docs/package_reference/sentence_transformer/SentenceTransformer.html)
> in the [Sentence Transformers documentation](https://sbert.net/). The model card describes the
> model, this describes the class you are importing and what `.encode` accepts.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# each split is encoded once and reused for the rest of the notebook
train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
val_embeddings = model.encode(data["validation"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)

print("train:", train_embeddings.shape)
print("validation:", val_embeddings.shape)
print("test:", test_embeddings.shape)

**Interpret the output.**

1. The shape has two numbers. Say where each one comes from, and which of the two would change if
   you swapped the embedding model.
2. Nothing about the labels was used in this cell. Say what that means for how often you would have
   to run it if you changed the task.
3. These vectors are the only thing the classifier will ever see. Name one thing about a review
   that survives this step and one thing that does not.

4. The model chose a pooling method for you. Say which of the Module 3 options you would pick if
   you had to choose yourself, and what you would expect to go wrong with a `[CLS]` vector taken
   from a model that was never trained to produce sentence vectors.

**Your answer:**

_Type here._

---

## Part 2 · A classifier on top

The classifier is ordinary logistic regression from scikit-learn. It takes the 768 numbers of a
review and learns weights that separate positive from negative. It never sees the text.

- **Python note:** `fit` trains on the training vectors and their labels. `predict` applies the
  learned weights to vectors it has not seen. `random_state` fixes the randomness so that a rerun
  gives the same answer.

> **Predict before running.**
>
> The model in Part 1 was trained by other people on tweets and scored around 0.80 F1 here.
>
> 1. Will a logistic regression trained on this dataset's own labels beat it?
> 2. What in the setup decides your answer, the classifier or the embeddings?

**Your prediction:**

_Type here._

In [ ]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(random_state=42, max_iter=1000)
classifier.fit(train_embeddings, data["train"]["label"])

y_pred = classifier.predict(test_embeddings)

evaluate(data["test"]["label"], y_pred, title="logistic regression")

**Interpret the output.**

1. Compare this against the task specific model from Part 1. Say which won and by how much, and
   whether the gap is large enough to matter.
2. The classifier trained in seconds. Say where the work that made this possible actually happened.
3. Suppose the gap had gone the other way. Name two things you would check before concluding that
   embeddings are the worse route.

**Your answer:**

_Type here._

---

## Part 3 · No labels at all

Every route so far used the training labels. The logistic regression learned from them, the class
averages were built from them, and even the task specific model in Part 1 used labels that somebody
else collected.

This one uses none. The embedding model turns any string into a vector, and it does not care
whether that string is a review. So write one sentence describing each class, encode those two
sentences, and they land in the same space as the reviews. A review is labelled by whichever
description it sits closer to.

The only thing you supply is the wording of the two descriptions.

> **Predict before running.**
>
> 1. The class averages in Part 3 were built from 8,530 labelled reviews. These two vectors come
>    from two sentences. How much do you expect to lose?
> 2. Name a class you could describe in one sentence, and one you could not.

**Your prediction:**

_Type here._

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# the two sentences are the only input to this method
label_descriptions = ["A negative review", "A positive review"]
label_embeddings = model.encode(label_descriptions)

print("one vector per label:", label_embeddings.shape)

In [ ]:
# compare every test review against both label descriptions, the closer one wins
zero_shot_similarity = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(zero_shot_similarity, axis=1)

evaluate(data["test"]["label"], y_pred, title="zero shot")

**Interpret the output.**

1. Hold this number. Part 4 builds two vectors from the labelled training set instead of from two
   sentences, and the gap between the two is exactly what 8,530 labels are worth here.
2. Change the descriptions to something longer or more specific and rerun the two cells above. Say
   what moved and what that tells you about how stable this method is.
3. This is the only method here that can add a new class without new data. Say what you would have
   to do to add a neutral class to each of the other routes.

**Your answer:**

_Type here._

---

## Part 4 · No classifier at all

Nothing forces you to train anything. If positive reviews really do land near each other, then the
average of all the positive training vectors is a point in the middle of that region, and the same
for negative. A review can be labelled by whichever average it sits closer to.

This is still supervised, because the labels decide which vectors go into which average. It just
has no model and no training step.

Closeness here is cosine similarity, the measure from Module 3: the angle between two vectors,
ignoring their length. One means the same direction, zero means unrelated.

- **Python note:** the loop below collects the vectors for one class into a list, then `np.mean`
  with `axis=0` averages them position by position, giving one vector of the same length.

> **Predict before running.**
>
> 1. Will this beat the trained logistic regression, lose badly, or land close?
> 2. Zero shot in Part 3 also compared against two vectors. These two are built from the labels.
>    How much do you expect that to be worth?

**Your prediction:**

_Type here._

In [ ]:
train_labels = data["train"]["label"]

# collect the training vectors belonging to each class
negative_vectors = []
positive_vectors = []

for i in range(len(train_labels)):
    if train_labels[i] == 0:
        negative_vectors.append(train_embeddings[i])
    else:
        positive_vectors.append(train_embeddings[i])

# one average vector per class
class_averages = np.array([
    np.mean(negative_vectors, axis=0),
    np.mean(positive_vectors, axis=0),
])

print("one average vector per class:", class_averages.shape)

# similarity of every test review to each of the two averages, then the closer one wins
similarities = cosine_similarity(test_embeddings, class_averages)
y_pred = np.argmax(similarities, axis=1)

evaluate(data["test"]["label"], y_pred, title="class averages")

**Interpret the output.**

1. Put this number next to the logistic regression and next to zero shot. Say what the labels
   bought over zero shot, and what training bought over averaging.
2. This method has no weights and no training. Say what it does still need, and what would break it.
3. Look at the normalized panel. Is the method equally good at both classes, and if not, propose a
   reason from how the averages were built.

**Your answer:**

_Type here._

---

## Part 5 · A vote of the nearest neighbours

A third option keeps every training vector instead of averaging them. To label a review, find the
k training vectors closest to it and take the majority vote.

Nothing is trained. The cost moves to prediction time, because every prediction compares against
the whole training set.

`k` is a choice, and that is what makes this section different from the two before it. The task
specific model was trained by somebody else. The logistic regression ran on defaults. Nothing had to
be decided, so nothing had to be checked.

Choosing k on the test split would raise the number you report and make it meaningless. Sweeping
five values and keeping the best is taking the maximum of five noisy measurements, and a maximum is
biased upward. On a thousand test reviews that noise is worth a point or two of F1, so the method
flatters itself by about that much, and it does so invisibly.

The validation split absorbs the choice. Spend the noise there, fix k, then measure on test once
with nothing left to decide. Rotten Tomatoes ships 8,530 training rows, 1,066 validation rows and
1,066 test rows for exactly this reason.

- **Python note:** `metric="cosine"` tells the neighbour search to use the angle between vectors
  rather than straight line distance, which matches how these embeddings are meant to be compared.

> **Predict before running.**
>
> 1. A very small k, say 1, against a large k, say 20. Which is more likely to overfit, and what
>    does overfitting look like in a method that never trains?
> 2. Would you expect this to beat the class averages from Part 4 and the zero shot route from Part 3?

**Your prediction:**

_Type here._

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

val_labels = data["validation"]["label"]

k_values = [1, 3, 5, 11, 21]
scores = []

for k in k_values:
    neighbours = KNeighborsClassifier(n_neighbors=k, metric="cosine")
    neighbours.fit(train_embeddings, train_labels)
    val_pred = neighbours.predict(val_embeddings)
    score = f1_score(val_labels, val_pred, average="macro")
    scores.append(score)
    print("k =", k, "validation macro F1 =", round(score, 3))

best_k = k_values[np.argmax(scores)]
print()
print("best k on validation:", best_k)

Now train with that k and measure on the test split, once.

In [ ]:
neighbours = KNeighborsClassifier(n_neighbors=best_k, metric="cosine")
neighbours.fit(train_embeddings, train_labels)

y_pred = neighbours.predict(test_embeddings)

evaluate(data["test"]["label"], y_pred, title="k nearest neighbours")

**Interpret the output.**

1. Look at how the validation score moved across the k values. Say whether the best k was an
   obvious winner or a coin toss, and what you would do if it was a coin toss.
2. The test number was measured once, after k was chosen. Say what would have been wrong with
   choosing k on the test split instead, given that it would have produced a higher number.
3. This route stores every training vector. Say what that costs you that the logistic regression
   does not.

**Your answer:**

_Type here._

---

## Part 6 · What the four routes cost

All four used the same vectors. The difference between them is only what happens after the
embedding model has done its work.

Fill this in from your own runs rather than from memory. The last row is the number you recorded
at the end of Part 1, so have that notebook open, or the figure you wrote down.

| Route | Macro F1 | Labels used | Cost at prediction time |
|---|---|---|---|
| Logistic regression | | all 8,530 | multiply by a weight vector |
| Class averages | | all 8,530, to build the averages | compare against two vectors |
| Zero shot | | none | compare against two vectors |
| Nearest neighbours | | all 8,530, kept rather than learned from | compare against every training vector |
| Task specific model, Part 1 | | somebody else's, on other text | one forward pass |

> **Your turn.**
>
> Rerun Part 2 with only the first 500 training reviews instead of all 8,530, and evaluate again.
> Then do the same for the class averages in Part 4. Which of the two degrades faster, and what
> does that tell you about which method needs more labelled data? Zero shot does not move at all
> under this test. Say why.

In [ ]:
# Your turn: retrain on a smaller slice of the training set and compare.

---

## Appendix · Comparing the four routes, optional

Everything above reported one number per method. This section is optional and looks at the same
four methods two other ways: an ROC curve, which shows how each one behaves at every threshold
rather than at the single cut it happens to use, and a box plot of macro F1 under resampling,
which shows how much of the gap between the methods is real and how much is noise.

Nothing is retrained here. The fitted objects from Parts 2 to 5 are still in memory and are
reused.

- **Python note:** `predict_proba` returns a probability per class instead of a hard label. The
  class averages have no probability, so the score used there is the similarity to the positive
  average minus the similarity to the negative one, which is positive when the review leans
  positive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity

test_labels = np.array(data["test"]["label"])

# one continuous score per method, higher means more positive
averages_similarity = cosine_similarity(test_embeddings, class_averages)

method_scores = {}
method_scores["logistic regression"] = classifier.predict_proba(test_embeddings)[:, 1]
method_scores["class averages"] = averages_similarity[:, 1] - averages_similarity[:, 0]
method_scores["zero shot"] = zero_shot_similarity[:, 1] - zero_shot_similarity[:, 0]
method_scores["nearest neighbours"] = neighbours.predict_proba(test_embeddings)[:, 1]

# the hard label each method actually produced
method_predictions = {}
method_predictions["logistic regression"] = classifier.predict(test_embeddings)
method_predictions["class averages"] = np.argmax(averages_similarity, axis=1)
method_predictions["zero shot"] = np.argmax(zero_shot_similarity, axis=1)
method_predictions["nearest neighbours"] = neighbours.predict(test_embeddings)

for name in method_scores:
    area = roc_auc_score(test_labels, method_scores[name])
    print(name, "AUC =", round(area, 3))

The ROC curve sweeps the threshold from one end to the other and plots the true positive rate
against the false positive rate. A method whose curve sits above another's is better at every
operating point, not only at the one it was measured at. The diagonal is what guessing looks like.

In [ ]:
plt.figure(figsize=(6, 6))

for name in method_scores:
    false_positive_rate, true_positive_rate, thresholds = roc_curve(
        test_labels, method_scores[name]
    )
    area = roc_auc_score(test_labels, method_scores[name])
    plt.plot(false_positive_rate, true_positive_rate, label=name + ", AUC " + str(round(area, 3)))

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="guessing")

plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC on the test split")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

The single macro F1 each method reported came from one test split of 1,066 reviews. A different
1,066 reviews would have given a slightly different number. Resampling the test split with
replacement and scoring again, many times, shows how wide that wobble is.

If two boxes overlap heavily, the gap between those two methods is inside the noise and is not
worth claiming.

- **Python note:** drawing 1,066 rows with replacement from the 1,066 we have is called a
  bootstrap. Some reviews appear twice, some not at all, which is what makes each round different.

In [ ]:
rng = np.random.default_rng(42)
rounds = 200

f1_samples = {}
for name in method_predictions:
    f1_samples[name] = []

for round_number in range(rounds):
    index = rng.integers(0, len(test_labels), len(test_labels))
    for name in method_predictions:
        predictions = np.array(method_predictions[name])
        score = f1_score(test_labels[index], predictions[index], average="macro")
        f1_samples[name].append(score)

names = []
samples = []
for name in f1_samples:
    names.append(name)
    samples.append(f1_samples[name])
    print(name, "median macro F1 =", round(float(np.median(f1_samples[name])), 3))

plt.figure(figsize=(7, 4))
plt.boxplot(samples, labels=names)
plt.ylabel("Macro F1")
plt.title("Macro F1 over 200 resamples of the test split")
plt.tight_layout()
plt.show()

**Interpret the output.**

1. Rank the three methods by AUC and by macro F1. Say whether the two rankings agree, and if they
   do not, say which one answers the question "is this the better method" and which answers "is
   this the better method at the threshold I am using".
2. Look at how far the boxes overlap. Name any pair of methods whose difference you would not
   report as real, and say what you would need in order to claim it.
3. One method's curve can sit above another's while its F1 is lower. Explain how that happens.

**Your answer:**

_Type here._

---

## Check yourself

- Say what is frozen in this route and what is trained.
- The embeddings were computed once and reused four times. Say why that is possible here and would
  not be if the embedding model were being fine tuned.
- Give the one sentence reason k was chosen on validation rather than test.
- Name the one thing all three methods in this notebook needed that Part 3 of the module will not.

## Summary

An embedding model turns text into vectors without knowing the task. A separate classifier turns
vectors into labels.

Because the embedding model is frozen, the vectors are computed once and reused by every method
tried afterwards.

A trained logistic regression, an average vector per class, two label sentences, and a neighbour
vote are four ways to use the same vectors, and they differ mostly in what they cost rather than in
what they score.

Hyperparameters like k are chosen on the validation split. A number measured on the test split
after tuning on it is not a number you can report.

Three of the four needed labels. The one that did not still came within a few points of them.

## References

- Alammar and Grootendorst, *Hands-On Large Language Models*, Chapter 4, Text Classification.
  This notebook is adapted from the book's own Chapter 4 notebook.
- Book code repository: [HandsOnLLM/Hands-On-Large-Language-Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models)
- The book's Chapter 4 notebook:
  [on GitHub](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter04/Chapter%204%20-%20Text%20Classification.ipynb)
  and [open in Colab](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter04/Chapter%204%20-%20Text%20Classification.ipynb)
- [Jay Alammar, A Visual Guide to Using BERT for the First Time](https://jalammar.github.io/a-visual-guide-to-using-bert-for-the-first-time/)
- Model card: [sentence-transformers/all-mpnet-base-v2](https://huggingface.co/sentence-transformers/all-mpnet-base-v2)
- Dataset card: [cornell-movie-review-data/rotten_tomatoes](https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes)
- [Sentence Transformers documentation](https://sbert.net/)
- [sklearn LogisticRegression reference](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [sklearn KNeighborsClassifier reference](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)